# CERES-Millet Experiment Notebook

End-to-end demonstration of the DSSAT-Python **CERES-Millet** model (`MLCER048`).

Pearl millet (*Pennisetum glaucum*) is the staple cereal of the semi-arid tropics.
CERES-Millet uses the same algorithm as CERES-Sorghum with a higher base temperature
(8 °C) and smaller individual kernel weight.

Topics:
1. Defining an experiment in JSON
2. Running a full season simulation
3. End-of-season summary
4. Daily trajectory plots
5. Cultivar CUL file I/O
6. Sensitivity analysis — effect of individual kernel weight (`g2`)
7. Comparison of two cultivars

## 1 — Setup

In [ ]:
import tempfile, os, pathlib
from dssat.io.converters.cul import write_cul, read_cul

cul_records = [
    {'VAR#': 'LCSA94', 'VRNAME': 'LCSA9401 Local', 'ECO#': 'DFAULT',
     'P1': 350.0, 'P2O': 12.5, 'P2R': 100.0,
     'P3': 80.0,  'P4':  40.0, 'P5':  400.0,
     'G1':   5.5, 'G2':   8.0, 'PHINT': 38.9},
    {'VAR#': 'BIDI00', 'VRNAME': 'Bidi Bala improved', 'ECO#': 'DFAULT',
     'P1': 380.0, 'P2O': 12.5, 'P2R': 120.0,
     'P3': 90.0,  'P4':  45.0, 'P5':  420.0,
     'G1':   6.0, 'G2':  10.0, 'PHINT': 38.9},
]

with tempfile.NamedTemporaryFile(suffix='.CUL', delete=False, mode='w') as f:
    cul_path = f.name

write_cul(cul_records, cul_path, model_code='ML')
print(pathlib.Path(cul_path).read_text())

read_back = read_cul(cul_path, model_code='ML')
print('Read back:', json.dumps(read_back, indent=2))
os.unlink(cul_path)

## 2 — Experiment JSON

In [ ]:
experiment = {
    "experiment_id": "MLEXP0001",
    "title": "CERES-Millet demo — Sahelian zone",
    "crop": "ML",
    "model": "MLCER048",

    # --- cultivar (LCSA9401 — local Sahelian variety)
    "cultivar": {
        "varno": "LCSA94",
        "vrname": "LCSA9401 Local",
        "p1":    350.0,
        "p2o":   12.5,
        "p2r":   100.0,
        "p3":     80.0,
        "p4":     40.0,
        "p5":    400.0,
        "g1":      5.5,
        "g2":      8.0,
        "phint":  38.9,
        "panth": 450.0,
        "tbase":   8.0,
        "topt":   33.0,
        "ropt":   28.0,
        "rue":     3.2
    },

    "planting": {
        "date": "2020-06-20",
        "plant_population": 8.0,
        "row_spacing": 80.0,
        "sowing_depth": 3.0
    },

    "simulation": {
        "start_date": "2020-06-20",
        "end_date":   "2020-11-30",
        "water_balance": false,
        "nitrogen_cycle": false
    },

    "soil": {
        "id": "NI00000001",
        "name": "Sahelian Entisol (loamy sand)",
        "country": "NE",
        "layers": [
            {"depth_cm":  20, "bd": 1.55, "ll": 0.040, "dul": 0.120, "sat": 0.32, "oc": 0.45, "ph": 5.8},
            {"depth_cm":  40, "bd": 1.60, "ll": 0.045, "dul": 0.130, "sat": 0.31, "oc": 0.25, "ph": 5.9},
            {"depth_cm":  60, "bd": 1.65, "ll": 0.050, "dul": 0.135, "sat": 0.30, "oc": 0.15, "ph": 6.0},
            {"depth_cm":  80, "bd": 1.65, "ll": 0.055, "dul": 0.140, "sat": 0.30, "oc": 0.10, "ph": 6.0},
            {"depth_cm": 100, "bd": 1.70, "ll": 0.058, "dul": 0.142, "sat": 0.29, "oc": 0.07, "ph": 6.1}
        ],
        "initial_sw_fraction": 0.5
    }
}

print(json.dumps(experiment, indent=2)[:500], '...')

## 3 — Run the simulation

In [ ]:
# Build soil
soil = SoilType()
layers = experiment['soil']['layers']
soil.nlayr = len(layers)
depth_cum = 0.0
for i, lyr in enumerate(layers):
    thick = lyr['depth_cm'] - depth_cum
    soil.dlayr[i] = thick
    soil.ds[i]    = lyr['depth_cm']
    soil.ll[i]    = lyr['ll']
    soil.dul[i]   = lyr['dul']
    soil.sat[i]   = lyr['sat']
    soil.bd[i]    = lyr['bd']
    soil.shf[i]   = 1.0
    soil.kg2ppm[i]= 10.0 / (lyr['bd'] * thick)
    depth_cum     = lyr['depth_cm']

sw_init = 0.5
sw_full = np.zeros(NL)
sw_full[:soil.nlayr] = [soil.dul[i]*sw_init + soil.ll[i]*(1-sw_init) for i in range(soil.nlayr)]

# Cultivar
cv_dict = experiment['cultivar']
cultivar = MilletCultivar(
    varno=cv_dict['varno'], vrname=cv_dict['vrname'],
    p1=cv_dict['p1'], p2o=cv_dict['p2o'], p2r=cv_dict['p2r'],
    p3=cv_dict['p3'], p4=cv_dict['p4'], p5=cv_dict['p5'],
    g1=cv_dict['g1'], g2=cv_dict['g2'], phint=cv_dict['phint'],
    panth=cv_dict['panth'], tbase=cv_dict['tbase'], topt=cv_dict['topt'],
    ropt=cv_dict.get('ropt', 28.0), rue=cv_dict.get('rue', 3.2),
)

yrsim = 2020172  # 2020-06-20 = DOY 172
pltg = experiment['planting']

model = CeresMillet(
    cultivar=cultivar,
    pltpop=pltg['plant_population'],
    sdepth=pltg['sowing_depth'],
    yrsim=yrsim,
    yrplt=yrsim,
)

iswitch = SwitchType()
iswitch.iswwat = 'N'
iswitch.iswnit = 'N'

ctrl = ControlType()
ctrl.yrsim = yrsim

w0 = WeatherType(); w0.tmax=38; w0.tmin=25; w0.srad=24; w0.dayl=13.5; w0.co2=380; w0.tavg=31.5

ctrl.dynamic = RUNINIT; ctrl.yrdoy = yrsim
model.run(ctrl, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
ctrl.dynamic = SEASINIT
model.run(ctrl, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

records = []
for day in range(1, 181):
    yrdoy = incdat(yrsim, day)
    ctrl.yrdoy = yrdoy
    doy = yrdoy % 1000

    # Sahel weather pattern: hot, rains June-Sept, dry season from Oct
    tmax = 40.0 - 0.06 * max(0, doy - 210)
    tmin = 25.0 - 0.04 * max(0, doy - 210)
    srad = 22.0 + 3.0 * np.cos((doy - 210) * np.pi / 100)
    dayl = 13.5 - 0.015 * max(0, doy - 172)

    w = WeatherType()
    w.tmax = max(28, tmax); w.tmin = max(18, tmin)
    w.srad = max(10, srad); w.dayl = max(11, dayl)
    w.co2 = 380.0; w.tavg = (w.tmax + w.tmin) / 2

    ctrl.dynamic = RATE
    model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl.dynamic = INTEGR
    model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    g = model.growth
    p = model.pheno
    records.append({
        'day': day, 'yrdoy': yrdoy, 'doy': doy,
        'istage': p.istage, 'xstage': p.xstage,
        'lai': g.lai, 'biomas': g.biomas, 'yield_kg_ha': g.yield_,
        'tmax': w.tmax, 'tmin': w.tmin, 'srad': w.srad,
    })
    if p.istage == 6 and p.mdate > 0:
        print(f'Maturity: day {day} (YRDOY {yrdoy})')
        break

df = pd.DataFrame(records)
print(f'{len(df)} days simulated')
print(df[['day','istage','lai','biomas','yield_kg_ha']].tail(8).to_string(index=False))

## 4 — End-of-season summary

In [ ]:
summary = model.summary()
print('\n=== CERES-Millet End-of-Season Summary ===')
for k, v in summary.items():
    print(f'  {k:<25} {v}')

## 5 — Daily trajectory plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('CERES-Millet — Daily Trajectories', fontsize=13, fontweight='bold')

stage_labels = {1:'E-juv',2:'Pan init',3:'Flag',4:'Anthesis',5:'Grain fill',6:'Maturity'}

ax = axes[0, 0]
ax.plot(df['day'], df['xstage'], color='darkorange', lw=2)
for stage, label in stage_labels.items():
    rows = df[df['istage'] == stage]
    if not rows.empty:
        ax.axvline(rows['day'].iloc[0], color='grey', ls='--', alpha=0.5)
        ax.text(rows['day'].iloc[0]+0.5, 0.5, label, fontsize=7, rotation=90, va='bottom')
ax.set_ylim(0, 7); ax.set_xlabel('Days after sowing')
ax.set_ylabel('Growth stage'); ax.set_title('Phenological development')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(df['day'], df['lai'], color='limegreen', lw=2)
ax.set_ylabel('LAI (m² m⁻²)'); ax.set_xlabel('Days after sowing')
ax.set_title('Leaf area index'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(df['day'], df['biomas'], color='sienna', lw=2)
ax.set_ylabel('Biomass (g m⁻²)'); ax.set_xlabel('Days after sowing')
ax.set_title('Above-ground dry matter'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax2 = ax.twinx()
ax.plot(df['day'], df['tmax'], 'r-', alpha=0.6, lw=1.5, label='Tmax')
ax.plot(df['day'], df['tmin'], 'b-', alpha=0.6, lw=1.5, label='Tmin')
ax2.bar(df['day'], df['srad'], color='gold', alpha=0.4, width=1, label='Srad')
ax.set_ylabel('Temperature (°C)'); ax2.set_ylabel('Srad (MJ m⁻²)')
ax.set_xlabel('Days after sowing'); ax.set_title('Weather inputs')
ax.legend(loc='upper right', fontsize=9); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('millet_trajectories.png', bbox_inches='tight')
plt.show()

## 6 — Cultivar CUL file I/O

In [ ]:
import tempfile, os, pathlib
from dssat.io.converters.cul import write_cul, read_cul

cul_records = [
    {'VAR#': 'LCSA94', 'VRNAME': 'LCSA9401 Local', 'ECO#': 'DFAULT',
     'P1': 350.0, 'P2O': 12.5, 'P2R': 100.0,
     'P3': 80.0,  'P4':  40.0, 'P5':  400.0,
     'G1':   5.5, 'G2':   8.0, 'PHINT': 38.9},
    {'VAR#': 'BIDI00', 'VRNAME': 'Bidi Bala improved', 'ECO#': 'DFAULT',
     'P1': 380.0, 'P2O': 12.5, 'P2R': 120.0,
     'P3': 90.0,  'P4':  45.0, 'P5':  420.0,
     'G1':   6.0, 'G2':  10.0, 'PHINT': 38.9},
]

with tempfile.NamedTemporaryFile(suffix='.CUL', delete=False, mode='w') as f:
    cul_path = f.name

write_cul(cul_records, cul_path, model_code='ML')
print(pathlib.Path(cul_path).read_text())

read_back = read_cul(cul_path, model_code='ML')
print('Read back:', json.dumps(read_back, indent=2))
os.unlink(cul_path)

## 7 — Sensitivity to kernel weight (G2)

In [ ]:
g2_values = [5.0, 6.5, 8.0, 10.0, 12.0, 15.0]
sens_results = []

for g2 in g2_values:
    cv = MilletCultivar(
        p1=350, p2o=12.5, p2r=100, p3=80, p4=40, p5=400,
        g1=5.5, g2=g2, phint=38.9, panth=450,
        tbase=8.0, topt=33.0, ropt=28.0, rue=3.2,
    )
    m = CeresMillet(cultivar=cv, pltpop=8.0, sdepth=3.0, yrsim=yrsim, yrplt=yrsim)

    ctrl3 = ControlType(); ctrl3.yrsim = yrsim
    ctrl3.dynamic = RUNINIT; ctrl3.yrdoy = yrsim
    m.run(ctrl3, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl3.dynamic = SEASINIT
    m.run(ctrl3, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    for day in range(1, 181):
        yrdoy = incdat(yrsim, day)
        ctrl3.yrdoy = yrdoy
        doy = yrdoy % 1000
        tmax_ = max(28, 40.0 - 0.06 * max(0, doy - 210))
        tmin_ = max(18, 25.0 - 0.04 * max(0, doy - 210))
        srad_ = max(10, 22.0 + 3.0 * np.cos((doy - 210) * np.pi / 100))
        w_ = WeatherType()
        w_.tmax=tmax_; w_.tmin=tmin_; w_.srad=srad_; w_.dayl=13; w_.co2=380; w_.tavg=(tmax_+tmin_)/2
        ctrl3.dynamic = RATE
        m.run(ctrl3, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        ctrl3.dynamic = INTEGR
        m.run(ctrl3, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        if m.pheno.istage == 6 and m.pheno.mdate > 0:
            break

    s = m.summary()
    sens_results.append({'g2_mg': g2, 'yield_kg_ha': s['yield_kg_ha'],
                         'grain_wt_mg': s['grain_wt_mg']})
    print(f'G2={g2:5.1f} mg  →  yield={s["yield_kg_ha"]:7.1f} kg/ha  grain_wt={s["grain_wt_mg"]:.2f} mg')

df_g2 = pd.DataFrame(sens_results)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_g2['g2_mg'], df_g2['yield_kg_ha'], 'o-', color='darkorange', lw=2, ms=8)
ax.set_xlabel('G2 — kernel weight (mg)')
ax.set_ylabel('Grain yield (kg ha⁻¹)')
ax.set_title('Millet yield sensitivity to kernel weight (G2)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('millet_g2_sensitivity.png', bbox_inches='tight')
plt.show()

## 8 — Two-cultivar comparison

In [ ]:
cultivars = {
    'LCSA9401 (local)':   MilletCultivar(p1=350, p2r=100, p5=400, g2=8.0),
    'Bidi Bala (improved)': MilletCultivar(varno='BIDI00', vrname='Bidi Bala improved',
                                           p1=380, p2r=120, p5=420, g2=10.0, g1=6.0),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Millet — cultivar comparison', fontsize=12, fontweight='bold')

colors = ['darkorange', 'steelblue']
for (name, cv), color in zip(cultivars.items(), colors):
    m = CeresMillet(cultivar=cv, pltpop=8.0, sdepth=3.0, yrsim=yrsim, yrplt=yrsim)
    ctrl4 = ControlType(); ctrl4.yrsim = yrsim
    ctrl4.dynamic = RUNINIT; ctrl4.yrdoy = yrsim
    m.run(ctrl4, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl4.dynamic = SEASINIT
    m.run(ctrl4, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    lai_traj, biom_traj = [], []
    for day in range(1, 181):
        yrdoy = incdat(yrsim, day)
        ctrl4.yrdoy = yrdoy
        doy = yrdoy % 1000
        tmax_ = max(28, 40.0 - 0.06 * max(0, doy - 210))
        tmin_ = max(18, 25.0 - 0.04 * max(0, doy - 210))
        srad_ = max(10, 22.0 + 3.0 * np.cos((doy - 210) * np.pi / 100))
        w_ = WeatherType()
        w_.tmax=tmax_; w_.tmin=tmin_; w_.srad=srad_; w_.dayl=13; w_.co2=380; w_.tavg=(tmax_+tmin_)/2
        ctrl4.dynamic = RATE
        m.run(ctrl4, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        ctrl4.dynamic = INTEGR
        m.run(ctrl4, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        lai_traj.append(m.growth.lai); biom_traj.append(m.growth.biomas)
        if m.pheno.istage == 6 and m.pheno.mdate > 0:
            break

    axes[0].plot(range(1, len(lai_traj)+1), lai_traj, color=color, lw=2, label=name)
    axes[1].plot(range(1, len(biom_traj)+1), biom_traj, color=color, lw=2, label=name)

for ax, title, ylabel in zip(
    axes,
    ['Leaf area index', 'Above-ground biomass'],
    ['LAI (m² m⁻²)', 'Biomass (g m⁻²)']
):
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel('Days after sowing')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('millet_cultivar_comparison.png', bbox_inches='tight')
plt.show()